In [1]:
import sys
from pathlib import Path
_r = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'config.py').exists())
sys.path.insert(0, str(_r / 'scripts'))

from notebook_init import setup
cfg, PATHS, POPULATIONS, HARD_FILTERS, SITUATIONAL_FILTERS, ML = setup()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import subprocess
from scipy.stats import chi2_contingency, entropy
from sklearn.metrics import mutual_info_score
from sklearn.preprocessing import LabelEncoder
from statsmodels.stats.multitest import fdrcorrection
from utils import count_variants
print("All imports successful!")
import os


Project root : /home/ibmelab/Projects/AISNP_Research
genomes_data : /mnt/data/aisnp_data/1000genomes/
output root  : output
All imports successful!


## Step 1: Load LD-Pruned Pfile Info

In [2]:
# Input: LD-pruned pfile from step 02
INPUT_PFILE = str(PATHS.PLINK_LD_PRUNED)
SAMPLES_CSV = str(PATHS.SAMPLES_CSV)
OUTPUT_DIR = str(PATHS.outputs_dir("03_vcf_to_matrix"))

print(f"Input pfile: {INPUT_PFILE}")
print(f"Samples CSV: {SAMPLES_CSV}")

# Count variants
n_variants = count_variants(INPUT_PFILE)
print(f"Variants after LD pruning: {n_variants}")

# Load sample information
samples_df = pd.read_csv(SAMPLES_CSV, header=None)
samples_df.columns = ["sample", "pop", "super_pop"]
print(f"\nSamples: {len(samples_df)}")
print(f"Populations: {samples_df['pop'].unique()}")
print(f"\nPopulation distribution:")
print(samples_df['pop'].value_counts())

Input pfile: /mnt/data/aisnp_data/1000genomes/outputs/02_situational_filtering/SEA_JPT_CN_LD_pruned
Samples CSV: /mnt/data/aisnp_data/1000genomes/SEA_JPT_CN_subpopulation_samples.csv
Variants after LD pruning: 614759

Samples: 504
Populations: ['CN' 'SEA' 'JPT']

Population distribution:
pop
CN     208
SEA    192
JPT    104
Name: count, dtype: int64


## Step 2: Export Genotypes to VCF and Convert to Matrix

In [3]:
from plink import run_plink2_command
from pathlib import Path

# Skip if matrix already exists
if Path(str(PATHS.GENOTYPE_MATRIX)).exists():
    print(f'Genotype matrix already exists: {PATHS.GENOTYPE_MATRIX}')
    print('Delete it to regenerate.')
else:
    vcf_output = str(PATHS.cache_dir('03_vcf_to_matrix') / 'temp_vcf')
    run_plink2_command(['--pfile', str(PATHS.PLINK_LD_PRUNED),
                        '--export', 'vcf',
                        '--out', vcf_output])
    print(f'VCF exported: {vcf_output}.vcf')


Running: plink2 --threads 16 --pfile /mnt/data/aisnp_data/1000genomes/outputs/02_situational_filtering/SEA_JPT_CN_LD_pruned --export vcf --out /mnt/data/aisnp_data/1000genomes/cache/03_vcf_to_matrix/temp_vcf
PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /mnt/data/aisnp_data/1000genomes/cache/03_vcf_to_matrix/temp_vcf.log.
Options in effect:
  --export vcf
  --out /mnt/data/aisnp_data/1000genomes/cache/03_vcf_to_matrix/temp_vcf
  --pfile /mnt/data/aisnp_data/1000genomes/outputs/02_situational_filtering/SEA_JPT_CN_LD_pruned
  --threads 16

Start time: Tue Jun  2 19:48:09 2026
15186 MiB RAM detected, ~7836 available; reserving 7593 MiB for main workspace.
Using up to 16 threads (change this with --threads).
504 samples (0 females, 0 males, 504 ambiguous; 504 founders) loaded from
/mnt/data/aisnp_data/1000genomes/outputs/02_situational_filtering/SEA_JPT_CN_LD_pruned.ps

In [4]:
# Parse VCF and create genotype matrix
def vcf_to_genotype_matrix(vcf_path):
    """
    Convert VCF to numeric genotype matrix (int8).
    0 = homozygous ref, 1 = heterozygous, 2 = homozygous alt, -1 = missing
    """
    genotypes = {}
    sample_names = []
    
    with open(vcf_path, 'r') as f:
        for line in tqdm(f, desc="Parsing VCF"):
            if line.startswith('##'):
                continue
            elif line.startswith('#CHROM'):
                fields = line.strip().split('\t')
                sample_names = fields[9:]
                continue
            
            fields = line.strip().split('\t')
            chrom = fields[0]
            pos = fields[1]
            snp_id = fields[2] if fields[2] != '.' else f"{chrom}:{pos}"
            
            genos = []
            for gt_field in fields[9:]:
                gt = gt_field.split(':')[0]
                if gt in ['0/0', '0|0']:
                    genos.append(0)
                elif gt in ['0/1', '1/0', '0|1', '1|0']:
                    genos.append(1)
                elif gt in ['1/1', '1|1']:
                    genos.append(2)
                else:
                    genos.append(-1)  # Missing
            
            genotypes[snp_id] = genos
    
    df = pd.DataFrame(genotypes, index=sample_names).astype(np.int8)
    return df

print("Converting VCF to genotype matrix...")
genotype_df = vcf_to_genotype_matrix(f"{vcf_output}.vcf")

mem_mb = genotype_df.memory_usage(deep=True).sum() / 1e6
print(f"\nGenotype matrix shape: {genotype_df.shape}, dtype: {genotype_df.dtypes.iloc[0]}, {mem_mb:.0f} MB")
print(f"Samples: {len(genotype_df)}")
print(f"SNPs: {len(genotype_df.columns)}")


Converting VCF to genotype matrix...


Parsing VCF: 614950it [00:53, 11450.73it/s]



Genotype matrix shape: (504, 614759), dtype: int8, 310 MB
Samples: 504
SNPs: 614759


In [5]:
# Add population labels
genotype_df = genotype_df.reset_index()
genotype_df = genotype_df.rename(columns={'index': 'sample'})

# Merge with population info
genotype_df = genotype_df.merge(samples_df[['sample', 'pop']], on='sample', how='left')

# Reorder columns
cols = ['sample', 'pop'] + [c for c in genotype_df.columns if c not in ['sample', 'pop']]
genotype_df = genotype_df[cols]

print(f"Dataset with labels: {genotype_df.shape}")
print(f"\nPopulation distribution:")
print(genotype_df['pop'].value_counts())

display(genotype_df.head())

Dataset with labels: (504, 614761)

Population distribution:
pop
CN     208
SEA    192
JPT    104
Name: count, dtype: int64


,sample,pop,"1:13116[b37]T,G","1:13273[b37]G,C","1:13289[b37]C,T","1:13543[b37]T,G","1:14599[b37]T,A","1:14933[b37]G,A","1:14975[b37]C,T","1:15777[b37]A,G",...,"22:51233283[b37]C,T","22:51234422[b37]T,C","22:51236100[b37]G,A","22:51237069[b37]T,C","22:51237125[b37]G,A","22:51237364[b37]A,G","22:51239678[b37]G,T","22:51240084[b37]G,C","22:51241386[b37]C,G","22:51244163[b37]A,G"
0,HG00403,CN,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,HG00404,CN,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2,HG00406,CN,0,0,0,0,0,0,0,1,...,0,0,0,0,0,1,0,0,0,0
3,HG00407,CN,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,HG00409,CN,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [6]:
# Save genotype matrix — read by 04a and downstream notebooks
import os, numpy as np
os.makedirs(str(PATHS.outputs_dir('03_vcf_to_matrix')), exist_ok=True)

snp_cols = [c for c in genotype_df.columns if c not in ['sample', 'pop']]
np.savez_compressed(
    str(PATHS.GENOTYPE_MATRIX),
    G=genotype_df[snp_cols].values,
    samples=np.array(genotype_df['sample'].values, dtype=str),
)
with open(str(PATHS.GENOTYPE_MATRIX_COLS), 'w') as f:
    f.write('\n'.join(snp_cols))
print(f'Saved: {PATHS.GENOTYPE_MATRIX}')
print(f'Shape: {genotype_df[snp_cols].shape}, dtype: int8')


Saved: /mnt/data/aisnp_data/1000genomes/outputs/03_vcf_to_matrix/genotype_matrix_with_pop.parquet  shape=(504, 614761)


In [7]:
# Cleanup temporary VCF
import os
for ext in ['.vcf', '.log']:
    tmp = str(PATHS.cache_dir('03_vcf_to_matrix') / 'temp_vcf') + ext
    if os.path.exists(tmp):
        os.remove(tmp)
        print(f'Removed: {tmp}')


Removed: /mnt/data/aisnp_data/1000genomes/cache/03_vcf_to_matrix/temp_vcf.vcf
Removed: /mnt/data/aisnp_data/1000genomes/cache/03_vcf_to_matrix/temp_vcf.log


## Summary

In [8]:
print('='*60)
print('VCF → GENOTYPE MATRIX')
print('='*60)
print(f'  Input pfile : {PATHS.PLINK_LD_PRUNED}')
print(f'  Output      : {PATHS.GENOTYPE_MATRIX}')
print(f'  Shape       : {genotype_df.shape}')
print(f'  Populations : {genotype_df["pop"].value_counts().to_dict()}')


VCF → GENOTYPE MATRIX
  Input pfile : /mnt/data/aisnp_data/1000genomes/outputs/02_situational_filtering/SEA_JPT_CN_LD_pruned
  Output      : /mnt/data/aisnp_data/1000genomes/outputs/03_vcf_to_matrix/genotype_matrix_with_pop.parquet
  Shape       : (504, 614761)
  Populations : {'CN': 208, 'SEA': 192, 'JPT': 104}
